# Forward Pass Debugging: Layer-by-Layer Reference

This notebook performs a forward pass through a .keras model one layer at a time, for the purpose of verifying low-level implementations in C or RISC-V.

At each step, it prints the output of the current layer. These values act as ground truth for validating manual implementations. The output of one layer is passed directly as the input to the next, preserving the inference flow.

This setup helps identify discrepancies between the high-level model and its low-level counterparts, allowing for precise, layer-specific debugging.


In [117]:
import os
import tensorflow as tf
import numpy as np
import random

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


## 1.0 Load and preprocess the data

In [118]:
(images, labels), _ = tf.keras.datasets.mnist.load_data()
images = images.astype("float32") / 255.0  # Normalize the images to [0, 1]
images = np.expand_dims(images, -1)  # Add channel dimension
labels = tf.keras.utils.to_categorical(labels, 10)  # One-hot encode the labels

## 2.0 Helper Functions

### 2.1 Function to Get a random image and label

Get a random image and label from the dataset. This function is used to generate a random input for the model.

In [119]:
def save_in_riscv_format(x, filename):
    with open(filename, 'w') as f:
        if len(x.shape) == 4:
            _, height, width, channels = x.shape
            for c in range(channels):
                for i in range(height):
                    row = [f"{x[0, i, j, c]:.6f}" for j in range(width)]
                    f.write(f".float " + ", ".join(row) + "\n")
                f.write("\n")
        
        elif len(x.shape) == 2:
            rows, cols = x.shape
            values = [f"{x[i, j]:.6f}" for i in range(rows) for j in range(cols)]
            f.write(f".float " + ", ".join(values) + "\n")
        
        else:
            raise ValueError("Unsupported shape")



def get_random_image(index=None, output_file="random_image.txt"):
    # Randomly select an image and its label
    if index is None:
        index = random.randint(0, len(images) - 1)
    image = images[index].squeeze()  # (28, 28)
    image = tf.expand_dims(image, axis=0)  # Add batch dimension (1, 28, 28)
    image = tf.expand_dims(image, axis=-1)  # Add channel dimension (1, 28, 28, 1)
    
    label = np.argmax(labels[index])  # Get label
    
    # Save the image to a .txt file with 3 decimal places
    with open(output_file, "w") as f:
        for row in image.numpy().squeeze():  # Convert tensor to numpy and remove extra dimensions
            row_str = ".float " + ", ".join(f"{val:.3f}" for val in row)  # Using commas to separate values
            f.write(row_str + "\n")
    
    return image, label

### 2.2 Function to Print a tensor

Prints the shape and values of a tensor in a readable format.

For 4D tensors (e.g., batches of images), it prints the values of the first sample,
channel by channel. For 2D tensors (e.g., dense layer outputs), it prints all values
row by row. Other shapes are not currently supported.

In [120]:
def print_shape_and_values(x):
    print(f"Shape: {x.shape}")
    
    if len(x.shape) == 4:
        _, height, width, channels = x.shape
        for c in range(channels):
            for i in range(height):
                row = [f"{x[0, i, j, c]:.3f}" for j in range(width)]
                print(", ".join(row))
            print()
    
    elif len(x.shape) == 2:
        rows, cols = x.shape
        for i in range(rows):
            row = [f"{x[i, j]:.3f}" for j in range(cols)]
            print(", ".join(row))
    else:
        print("Unsupported shape")


## 3.0 Load the model from mnist_cnn_model.keras

In [121]:
model = tf.keras.models.load_model("../models/mnist_cnn_model.keras")

## 4.0 Get a random image, label and step through the model layer by layer

### 4.1 Get a random image and label

In [122]:
image, label = get_random_image(5)
print(f"Label: {label}")

Label: 2


### 4.2 Step through the model layer by layer

#### 4.2.1 Input Image

In [123]:
print("Original Image:")
print_shape_and_values(image)

Original Image:
Shape: (1, 28, 28, 1)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 

#### 4.2.2 Conv2D Layer

Note: Refer to section **4.2.1** for the input

In [124]:
conv2d_out = model.layers[0](image)
print_shape_and_values(conv2d_out)
save_in_riscv_format(conv2d_out, "conv2d_out.txt")

Shape: (1, 24, 24, 8)
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.447, -0.425, -0.348, -0.271, -0.393, -0.499, -0.489, -0.498, -0.462, -0.460, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.428, -0.289, -0.147, -0.095, -0.057, -0.085, -0.176, -0.327, -0.403, -0.449, -0.502, -0.460, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.421, -0.282, -0.109, -0.031, -0.039, -0.012, 0.030, 0.156, 0.264, 0.180, -0.010, -0.225, -0.436, -0.472, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.445, -0.302, -0.103, -0.036, -0.027, -0.060, -0.079, -0.210, -0.313, -0.014, 0.338, 0.286, 0.127, -0.123, -0.330, -0.453, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460

#### 4.2.3 ReLU Activation

Note: Refer to section **4.2.2** for the input

In [129]:
relu_out = model.layers[1](conv2d_out)
print_shape_and_values(relu_out)
save_in_riscv_format(relu_out, "relu_out.txt")

Shape: (1, 24, 24, 8)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.030, 0.156, 0.264, 0.180, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.338, 0.286, 0.127, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.001, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.0

#### 4.2.4 MaxPooling

Note: Refer to section **4.2.3** for the input

In [130]:
maxpool_out = model.layers[2](relu_out)
print_shape_and_values(maxpool_out)
save_in_riscv_format(maxpool_out, "maxpool_out.txt")

Shape: (1, 12, 12, 8)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.030, 0.264, 0.180, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.001, 0.000, 0.000, 0.000, 0.338, 0.286, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.022, 0.000, 0.039, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.195, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.031, 0.071, 0.000, 0.273, 0.043, 0.157, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.017, 0.000, 0.000, 0.203, 0.284, 0.518, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.205, 0.473, 0.141, 0.000, 0.000, 0.218, 0.360, 0.217, 0.022
0.000, 0.000, 0.597, 0.643, 0.139, 0.000, 0.110, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.282, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.

#### 4.2.5 Flatten

Note: Refer to section **4.2.4** for the input

In [131]:
flatten_out = model.layers[3](maxpool_out)
print_shape_and_values(flatten_out)
save_in_riscv_format(flatten_out, "flatten_out.txt")

Shape: (1, 1152)
0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.195, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.357, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.515, 0.000, 0.000, 0.062, 0.000, 0.000, 0.000, 0.000, 0.542, 0.000, 0.000, 0.071, 0.000, 0.000, 0.000, 0.000, 0.418, 0.000, 0.115, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.231, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.778, 0.000, 0.000, 0.034, 0.140, 0.209, 0.000, 0.000, 1.830, 0.000, 0.000, 0.1

#### 4.2.6 Fifth Layer: Dense Layer

Note: Refer to section **4.2.5** for the input

In [132]:
dense_out = model.layers[4](flatten_out)  # Fifth layer output
print_shape_and_values(dense_out)
save_in_riscv_format(dense_out, "dense_out.txt")

Shape: (1, 10)
-13.057, -9.941, 8.739, -6.403, -8.514, -13.468, -19.009, -6.483, -2.546, -10.781


#### 4.2.7 Layer Six: Softmax

Note: Refer to section **4.2.6** for the input

In [133]:
softmax_out = model.layers[5](dense_out)
print_shape_and_values(softmax_out)
save_in_riscv_format(softmax_out, "softmax_out.txt")

Shape: (1, 10)
0.000, 0.000, 1.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000


### 4.3 Get model prediction

In [116]:
print(f"Predicted class: {np.argmax(softmax_out)}")
print(f"True class: {label}")

Predicted class: 2
True class: 2
